# Setup Silver tables

In [0]:
from utils.logger import get_logger
from etl_config.silver_config import (
    CATALOG,
    SILVER_CONFIG,
    SILVER_SCHEMA,
    SILVER,
)

logger = get_logger("silver_setup")

# spark sql type -> DDL type mapping
TYPE_MAP = {
    "integer":       "INT",
    "string":        "STRING",
    "date":          "DATE",
    "decimal(10,2)": "DECIMAL(10,2)",
    "decimal(10,4)": "DECIMAL(10,4)",
    "decimal(3,2)":  "DECIMAL(3,2)",
}

In [0]:
for table_name, cfg in SILVER_CONFIG.items():

    col_ddl = ",\n        ".join(
        f"{col_name} {col_type}" + (" NOT NULL" if col_name in cfg.required_columns else "")
        for col_name, col_type in cfg.cast_columns.items()
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {cfg.target_table} (
            {col_ddl}
        )
        USING DELTA
        COMMENT '{cfg.table_description}'
    """)
    logger.info(f"Table created/verified: {cfg.target_table}")

    spark.sql(f"""
        ALTER TABLE {cfg.target_table}
        SET TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
    """)

    for col_name, comment in cfg.column_comments.items():
        spark.sql(f"""
            COMMENT ON COLUMN {cfg.target_table}.{col_name}
            IS '{comment}'
        """)

    logger.info(f"Metadata applied for: {cfg.target_table}")

### Watermark table

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}._watermark (
        table_name   STRING    NOT NULL,
        batch_id     STRING    NOT NULL,
        ingested_at  TIMESTAMP,
        processed_at TIMESTAMP,
        rows_merged  LONG,
        status       STRING    NOT NULL
    )
    USING DELTA
    COMMENT 'Tracking of processed Bronze batches per table'
""")
logger.info(f"Watermark table created/verified: {SILVER}._watermark")

spark.sql(f"""
    ALTER TABLE {SILVER}._watermark
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")
logger.info(f"Table properties applied for: {SILVER}._watermark")

In [0]:
# quarantine table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}._quarantine (
        table_name       STRING    NOT NULL,
        batch_id         STRING,
        rejected_at      TIMESTAMP,
        error_type       STRING    NOT NULL,
        _missing_columns ARRAY<STRING>,
        raw_data         STRING
    )
    USING DELTA
    COMMENT 'Bad rows due to missing or uncastable values'
""")
logger.info(f"Quarantine table created/verified: {SILVER}._quarantine")

spark.sql(f"""
    ALTER TABLE {SILVER}._quarantine
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")
logger.info(f"Table properties applied for: {SILVER}._quarantine")